In [ ]:
from scipy.stats import spearmanr
import pandas as pd

from scipy.stats import spearmanr
import pandas as pd

def evaluate_comprehensive_alignment(df_string, df_diffstat, dataset_name, threshold=10000):
    metrics = ['lines_added', 'lines_removed', 'lines_modified']
    
    comparison_df = pd.merge(
        df_string[['commit_id'] + metrics], 
        df_diffstat[['commit_id'] + metrics], 
        on='commit_id', 
        suffixes=('_string', '_diffstat')
    )

    comparison_df = comparison_df[comparison_df['lines_modified_diffstat'] <= threshold].dropna()

    results = {}
    print(f"=== {dataset_name} Detailed Comparison (N={len(comparison_df)}) ===")
    
    for m in metrics:
        s_col = f"{m}_string"
        d_col = f"{m}_diffstat"
        
        rho, _ = spearmanr(comparison_df[s_col], comparison_df[d_col])
        mae = (comparison_df[s_col] - comparison_df[d_col]).abs().mean()
        med_ae = (comparison_df[s_col] - comparison_df[d_col]).abs().median()
        
        results[m] = {'rho': rho, 'mae': mae}
        
        print(f"Metric: {m:15}")
        print(f"  - Spearman's Rho:   {rho:.4f}")
        print(f"  - Mean Abs Error:   {mae:.2f}")
        print(f"  - Median Abs Error: {med_ae:.2f}")
    print("=" * 45 + "\n")
    
    return results, comparison_df

def get_manageable_conflicts(comp_df, max_lines=30, n=15):
    """
    Finds conflicts in small, human-readable commits.
    """
    manageable = comp_df[comp_df['lines_modified_diffstat'] <= max_lines].copy()
    return manageable.sort_values(by='abs_error', ascending=False).head(n)

df_icvul_string = pd.read_csv('../data/intermediate/churn_icvul_semantic.csv')
df_icvul_diffstat = pd.read_csv('../data/intermediate/churn_icvul.csv')

df_linux_string = pd.read_csv('../data/intermediate/churn_linux_semantic.csv')
df_linux_diffstat = pd.read_csv('../data/intermediate/churn_linux.csv')

df_ds_apache_string = pd.read_csv('../data/intermediate/churn_ds_apache_semantic.csv')
df_ds_apache_diffstat = pd.read_csv('../data/intermediate/churn_ds_apache.csv')

df_tosem_string = pd.read_csv('../data/intermediate/churn_tosem_semantic.csv')
df_tosem_diffstat = pd.read_csv('../data/intermediate/churn_tosem.csv')

icvul_rho, icvul_comp_df = evaluate_comprehensive_alignment(df_icvul_string, df_icvul_diffstat, "ICVul")

linux_rho, linux_comp_df = evaluate_comprehensive_alignment(df_linux_string, df_linux_diffstat, "Linux")

tosem_rho, tosem_comp_df = evaluate_comprehensive_alignment(df_tosem_string, df_tosem_diffstat, "TOSEM")

ds_apache_rho, ds_apache_comp_df = evaluate_comprehensive_alignment(df_ds_apache_string, df_ds_apache_diffstat, "DS_APACHE")

=== ICVul Detailed Comparison (N=958) ===
Metric: lines_added    
  - Spearman's Rho:   0.9677
  - Mean Abs Error:   10.65
  - Median Abs Error: 1.00
Metric: lines_removed  
  - Spearman's Rho:   0.8646
  - Mean Abs Error:   10.62
  - Median Abs Error: 1.00
Metric: lines_modified 
  - Spearman's Rho:   0.9703
  - Mean Abs Error:   10.62
  - Median Abs Error: 1.00

=== Linux Detailed Comparison (N=9048) ===
Metric: lines_added    
  - Spearman's Rho:   0.9733
  - Mean Abs Error:   2.80
  - Median Abs Error: 0.00
Metric: lines_removed  
  - Spearman's Rho:   0.8491
  - Mean Abs Error:   2.80
  - Median Abs Error: 0.00
Metric: lines_modified 
  - Spearman's Rho:   0.9352
  - Mean Abs Error:   2.80
  - Median Abs Error: 0.00

=== TOSEM Detailed Comparison (N=275) ===
Metric: lines_added    
  - Spearman's Rho:   0.9905
  - Mean Abs Error:   6.69
  - Median Abs Error: 1.00
Metric: lines_removed  
  - Spearman's Rho:   0.9022
  - Mean Abs Error:   6.69
  - Median Abs Error: 1.00
Metric: line